In [1]:
# %pip install pandas numpy scikit-learn skl2onnx onnxruntime

   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/7.9 MB 839.0 kB/s eta 0:00:09
   -- ------------------------------------- 0.5/7.9 MB 839.0 kB/s eta 0:00:09
   --- ------------------------------------ 0.8/7.9 MB 960.8 kB/s eta 0:00:08
   ----- ---------------------------------- 1.0/7.9 MB 992.7 kB/s eta 0:00:07
   ------ --------------------------------- 1.3/7.9 MB 1.0 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/7.9 MB 1.1 MB/s eta 0:00:06
   --------- ------------------------------ 1.8/7.9 MB 1.1 MB/s eta 0:00:06
   ----------- ---------------------------- 2.4/7.9 MB 1.2 MB/s eta 0:00:05
   ------------- -------------------------- 2.6/7.9 MB 1.3 MB/s eta 0:00:05
   --------------- ------------------------ 3.1/7.9 MB 1.3 MB/s eta 0:00:04
   ----------------- ----


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\arkad\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skl2onnx import convert_sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from skl2onnx.common.data_types import FloatTensorType
from sklearn.metrics import classification_report, accuracy_score

url = "https://raw.githubusercontent.com/Gladiator07/Harvestify/master/Data-processed/crop_recommendation.csv"
df = pd.read_csv(url)

df = df.rename(columns={
    'temperature': 'T2M',
    'humidity': 'RH2M',
    'rainfall': 'PRECTOTCORR'
})

# 1) T2M_MAX и T2M_MIN: суточный разброс температур от влажности
diurnal_range = 14.0 * (1.0 - (df['RH2M'] / 100.0) * 0.5)
df['T2M_MAX'] = (df['T2M'] + diurnal_range / 2.0).round(2)
df['T2M_MIN'] = (df['T2M'] - diurnal_range / 2.0).round(2)

# 2) ALLSKY_SFC_SW_DWN: солнечная радиация по формуле Харгривза
delta_t = np.maximum(df['T2M_MAX'] - df['T2M_MIN'], 1.0)
extraterrestrial_rad = 32.0
hargreaves_radiation = 0.16 * np.sqrt(delta_t) * extraterrestrial_rad
rain_penalty = np.exp(-df['PRECTOTCORR'] / 400.0)
df['ALLSKY_SFC_SW_DWN'] = (hargreaves_radiation * rain_penalty).clip(8.0, 28.5).round(2)

# 3) GDD: градусо-дни (база 10°C за 120 дней вегетации)
df['GDD'] = (np.maximum(df['T2M'] - 10.0, 0.0) * 120).round(1)

# 4) WS2M: скорость ветра на высоте 2 м
np.random.seed(42)
df['WS2M'] = (2.2 + 0.03 * (df['T2M_MAX'] - df['T2M']) + np.random.normal(0, 0.4, len(df))).clip(1.2, 5.5).round(2)

feature_names = [
    'N', 'P', 'K', 'ph',
    'T2M', 'T2M_MAX', 'T2M_MIN',
    'RH2M', 'PRECTOTCORR',
    'ALLSKY_SFC_SW_DWN', 'GDD', 'WS2M'
]

# Float32 обязателен для корректной работы WebAssembly в браузере
X = df[feature_names].astype(np.float32)
y = df['label']

# Сохраняем список культур в алфавитном порядке (так их индексирует sklearn)
classes = sorted(y.unique().tolist())
# with open("classes.json", "w", encoding="utf-8") as f:
#     json.dump(classes, f, indent=2)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_model = RandomForestClassifier(
    n_estimators=100,      # 100 независимых деревьев
    max_depth=12,          # Ограничение глубины против переобучения и раздувания файла
    min_samples_split=4,   # Запрет дробить узлы, где меньше 4 строк
    min_samples_leaf=2,    # В каждом конечном листе должно остаться минимум 2 строки
    random_state=42,       # Фиксация генератора случайных чисел
    n_jobs=-1              # Параллельное обучение на всех ядрах CPU
)

rf_model.fit(X_train, y_train)

# Проверка точности на отложенной выборке
y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))

# # -------------------------------------------------------------
# # 4. Конвертация в ONNX для запуска в браузере (onnxruntime-web)
# # -------------------------------------------------------------
# # Задаем входной тензор: [размер_батча, 12_признаков]
# initial_type = [('float_input', FloatTensorType([None, len(feature_names)]))]

# # zipmap=False отключает словари Python и заставляет модель возвращать чистый массив float32 вероятностей
# onnx_model = convert_sklearn(
#     rf_model,
#     initial_types=initial_type,
#     options={id(rf_model): {'zipmap': False}}
# )

# onnx_filename = "crop_rf_model.onnx"
# with open(onnx_filename, "wb") as f:
#     f.write(onnx_model.SerializeToString())

# print(f"Готово! Модель сохранена в {onnx_filename}, маппинг классов — в classes.json")

Accuracy: 99.09%

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      0.95      0.97        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.95      1.00      0.98        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        20
      pap

In [8]:
df = pd.read_csv("seeds_rename.csv")

# df = df.rename(columns={
#     'temperature': 'T2M',
#     'humidity': 'RH2M',
#     'rainfall': 'PRECTOTCORR'
# })

X = df[feature_names]
y = df['label']

# df.to_csv('seeds_rename.csv', index=False)

# Сохраняем список культур в алфавитном порядке (так их индексирует sklearn)
# classes = sorted(y.unique().tolist())
# with open("classes.json", "w", encoding="utf-8") as f:
#     json.dump(classes, f, indent=2)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf_model = RandomForestClassifier(
    n_estimators=100,      # 100 независимых деревьев
    max_depth=12,          # Ограничение глубины против переобучения и раздувания файла
    min_samples_split=4,   # Запрет дробить узлы, где меньше 4 строк
    min_samples_leaf=2,    # В каждом конечном листе должно остаться минимум 2 строки
    random_state=42,       # Фиксация генератора случайных чисел
    n_jobs=-1              # Параллельное обучение на всех ядрах CPU
)

rf_model.fit(X_train, y_train)

# Проверка точности на отложенной выборке
y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred))

initial_type = [('float_input', FloatTensorType([None, len(feature_names)]))]

# zipmap=False отключает словари Python и заставляет модель возвращать чистый массив float32 вероятностей
onnx_model = convert_sklearn(
    rf_model,
    initial_types=initial_type,
    options={id(rf_model): {'zipmap': False}}
)

onnx_filename = "crop_rf_model.onnx"
with open(onnx_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Готово! Модель сохранена в {onnx_filename}, маппинг классов — в classes.json")

Accuracy: 99.55%

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      1.00      1.00        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      0.95      0.97        20
       maize       1.00      1.00      1.00        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.95      1.00      0.98        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        20
      pap

In [10]:
import requests
import numpy as np
import json
import onnxruntime as ort

session = ort.InferenceSession("crop_rf_model.onnx")
with open("classes.json", "r", encoding="utf-8") as f:
    classes = json.load(f)

lat, lon = 49.58, 34.55

url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "T2M,T2M_MAX,T2M_MIN,RH2M,PRECTOTCORR,ALLSKY_SFC_SW_DWN,WS2M",
    "community": "AG",
    "longitude": lon,
    "latitude": lat,
    "start": "20250501",
    "end": "20250831",
    "format": "JSON"
}

print(f"Запрос метеоданных NASA для координат [{lat}, {lon}]...")
response = requests.get(url, params=params, timeout=30)
data = response.json()

props = data["properties"]["parameter"]

t2m = np.mean(list(props["T2M"].values()))
t2m_max = np.mean(list(props["T2M_MAX"].values()))
t2m_min = np.mean(list(props["T2M_MIN"].values()))
rh2m = np.mean(list(props["RH2M"].values()))
# Осадки (PRECTOTCORR) обычно суммируются за сезон
prectotcorr = np.sum(list(props["PRECTOTCORR"].values())) / 4.0  # Среднемесячные !!!
allsky = np.mean(list(props["ALLSKY_SFC_SW_DWN"].values()))
ws2m = np.mean(list(props["WS2M"].values()))

# Накопленное тепло GDD (база 10°C) за сезон
daily_t2m = np.array(list(props["T2M"].values()))
gdd = float(np.sum(np.maximum(daily_t2m - 10.0, 0.0)))

print(f"\nКлиматический профиль NASA:")
print(f"  T2M: {t2m:.1f}°C (Макс: {t2m_max:.1f}°C, Мин: {t2m_min:.1f}°C)")
print(f"  Влажность: {rh2m:.1f}%")
print(f"  Сумма осадков: {prectotcorr:.1f} мм")
print(f"  GDD: {gdd:.1f}")
print(f"  Радиация: {allsky:.1f} МДж/м²")
print(f"  Ветер: {ws2m:.1f} м/с")

N = 50.0
P = 40.0
K = 45.0
ph = 6.8

input_vector = np.array([[
    N, P, K, ph,
    t2m, t2m_max, t2m_min,
    rh2m, prectotcorr,
    allsky, gdd, ws2m
]], dtype=np.float32)

input_name = session.get_inputs()[0].name
outputs = session.run(None, {input_name: input_vector})

# outputs[1] содержит вероятности [1, 22]
probs = outputs[1][0]

top_indices = np.argsort(probs)[::-1][:3]

print("\n--- Результат предсказания модели ---")
for rank, idx in enumerate(top_indices, 1):
    print(f"{rank}. {classes[idx]}: {probs[idx] * 100:.2f}%")

Запрос метеоданных NASA для координат [49.58, 34.55]...

Климатический профиль NASA:
  T2M: 20.3°C (Макс: 26.3°C, Мин: 13.9°C)
  Влажность: 57.3%
  Сумма осадков: 52.8 мм
  GDD: 1276.6
  Радиация: 20.7 МДж/м²
  Ветер: 2.9 м/с

--- Результат предсказания модели ---
1. mothbeans: 30.60%
2. lentil: 23.47%
3. pomegranate: 9.50%
